In [4]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        if filename.endswith(".joblib"):
            print(os.path.join(dirname, filename))

/kaggle/input/models/danilzhukovv/ecup-baseline-logreg-l12/scikitlearn/default/1/baseline_logreg_l12.joblib


In [ ]:
# ECUP CE v2 — Polars + 2x T4
import os,time,math,random,json,shutil,gc
from pathlib import Path
from collections import Counter
os.environ["TOKENIZERS_PARALLELISM"]="false"

import numpy as np,pandas as pd,polars as pl,pyarrow.parquet as pq,torch
import torch.nn.functional as F
from torch.utils.data import Dataset,IterableDataset,DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_cosine_schedule_with_warmup

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

BASE="/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"
ITEMS=f"{BASE}/items.parquet"; LLM=f"{BASE}/matches_llm.parquet"
INIT="/kaggle/input/models/danilzhukovv/ecup-product-matching-rubert/pytorch/default/1"
WORK=Path("/kaggle/working/ecup_ce_v2"); WORK.mkdir(exist_ok=True)
SPLIT=WORK/"split.parquet"; PAIRS=WORK/"pairs_text.parquet"; BEST=WORK/"best.pt"; OUT=WORK/"model"
MAXLEN=192; BS=256; EBS=512; LR=5e-5; HARD=1.5; EVAL=5000

assert torch.cuda.device_count()==2
assert Path(INIT,"model.safetensors").exists()
DEV=torch.device("cuda:0")
print([torch.cuda.get_device_name(i) for i in range(2)])

def component_mask(a,b,frac=.03,seed=13):
    parent={}
    def find(x):
        parent.setdefault(x,x)
        while x!=parent[x]: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y: parent[y]=x
    comp=np.fromiter((find(x) for x in a),np.int64,len(a))
    rng=np.random.RandomState(seed); u=np.unique(comp)
    chosen=set(u[rng.rand(len(u))<frac].tolist())
    return np.fromiter((x in chosen for x in comp),bool,len(a))

# ---------- split ----------
if not SPLIT.exists():
    t=time.time()
    m=pl.read_parquet(LLM,columns=["id1","id2","target"])
    mask=component_mask(m["id1"].to_numpy(),m["id2"].to_numpy())
    m.with_columns(pl.Series("is_val",mask)).write_parquet(SPLIT,compression="zstd")
    del m,mask; gc.collect()
    print(f"split ready: {(time.time()-t)/60:.1f} min")

# ---------- Polars text + joins ----------
if not PAIRS.exists():
    t=time.time()
    attrs=pl.col("attributes").cast(pl.String).fill_null("")
    cat=pl.col("category").cast(pl.String).fill_null("")
    important=attrs.str.extract_all(
        r'(?i)"[^"]*(?:бренд|brand|артикул|партномер|oem|sku|модель|размер|size|цвет|color|объем|обьем|вес|масса|количество|материал|тип)[^"]*"\\s*:\\s*"[^"]*"'
    ).list.join(" ; ")

    items=pl.scan_parquet(ITEMS).select(
        "id",
        cat.alias("category"),
        pl.concat_str([
            pl.col("name").cast(pl.String).fill_null(""),
            pl.lit(" | "),important,
            pl.lit(" | "),attrs.str.slice(0,260),
            pl.lit(" | категория:"),cat
        ]).alias("text")
    )
    i1=items.rename({"id":"id1","text":"text1","category":"cat1"})
    i2=items.rename({"id":"id2","text":"text2","category":"cat2"})

    (pl.scan_parquet(SPLIT)
      .join(i1,on="id1",how="inner")
      .join(i2,on="id2",how="inner")
      .filter(pl.col("cat1")==pl.col("cat2"))
      .select("text1","text2",pl.col("cat1").alias("category"),"target","is_val")
      .sink_parquet(PAIRS,compression="zstd",row_group_size=100_000))
    print(f"Polars pairs ready: {(time.time()-t)/60:.1f} min")

stats=(pl.scan_parquet(PAIRS).filter(~pl.col("is_val"))
       .group_by("category").len().collect())
N=int(stats["len"].sum()); K=len(stats)
CW={str(c):float(np.clip(N/(K*n),.65,1.6)) for c,n in zip(stats["category"],stats["len"])}

va=(pl.scan_parquet(PAIRS)
    .filter(pl.col("is_val") & ((pl.col("target")<=.2)|(pl.col("target")>=.8)))
    .with_columns((pl.col("target")>=.5).cast(pl.Float32))
    .collect().to_pandas())
print(f"train={N:,}, val={len(va):,}")

tok=AutoTokenizer.from_pretrained(INIT)

class TrainDS(IterableDataset):
    def __iter__(self):
        pf=pq.ParquetFile(PAIRS); rng=np.random.default_rng(SEED)
        groups=list(range(pf.num_row_groups)); rng.shuffle(groups)
        for rg in groups:
            for b in pf.iter_batches(row_groups=[rg],batch_size=8192):
                d=b.to_pydict()
                ix=np.flatnonzero(~np.asarray(d["is_val"],bool)); rng.shuffle(ix)
                for i in ix:
                    c=str(d["category"][i])
                    yield d["text1"][i],d["text2"][i],float(d["target"][i]),CW.get(c,1.)

class EvalDS(Dataset):
    def __init__(self,d): self.a=d.text1.tolist(); self.b=d.text2.tolist(); self.y=d.target.to_numpy(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.a[i],self.b[i],self.y[i],1.

def pack(rows,train=False):
    a,b,y,w=map(list,zip(*rows))
    if train:
        for i,s in enumerate(np.random.rand(len(a))<.5):
            if s:a[i],b[i]=b[i],a[i]
    z=tok(a,b,padding=True,truncation="longest_first",max_length=MAXLEN,return_tensors="pt")
    return z,torch.tensor(y),torch.tensor(w)

model=torch.nn.DataParallel(
    AutoModelForSequenceClassification.from_pretrained(INIT,num_labels=1).to(DEV),
    device_ids=[0,1]
)
core=lambda:model.module

def macro(d,p):
    z=d.reset_index(drop=True)
    return float(np.mean([average_precision_score(g.target,p[g.index]) for _,g in z.groupby("category")]))

@torch.inference_mode()
def predict(d):
    model.eval(); out=[]
    for z,_,_ in DataLoader(EvalDS(d),batch_size=EBS,num_workers=0,collate_fn=lambda x:pack(x)):
        z={k:v.to(DEV) for k,v in z.items()}
        with torch.autocast("cuda",dtype=torch.float16):
            out.append(torch.sigmoid(model(**z).logits.squeeze(-1).float()).cpu().numpy())
    return np.concatenate(out)

def predict_tta(d):
    p=predict(d); r=d.copy()
    r["text1"],r["text2"]=d["text2"],d["text1"]
    return (p+predict(r))/2

fast=pd.concat([g.sample(min(3000,len(g)),random_state=SEED) for _,g in va.groupby("category")]).reset_index(drop=True)
best=macro(fast,predict(fast))
torch.save({"model":core().state_dict(),"metric":best,"step":0},BEST)
print("initial macro:",best)

steps=N//BS
dl=DataLoader(TrainDS(),batch_size=BS,drop_last=True,num_workers=0,collate_fn=lambda x:pack(x,True))
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=.01)
sch=get_cosine_schedule_with_warmup(opt,int(steps*.03),steps)
scaler=torch.amp.GradScaler("cuda"); started=time.time(); run=0.

for step,(z,y,cw) in enumerate(dl,1):
    model.train(); z={k:v.to(DEV) for k,v in z.items()}; y=y.to(DEV); cw=cw.to(DEV)
    with torch.autocast("cuda",dtype=torch.float16):
        logits=model(**z).logits.squeeze(-1)
        base=F.binary_cross_entropy_with_logits(logits,y,reduction="none")
        with torch.no_grad():
            p=torch.sigmoid(logits.float())
            w=cw*(.5+3*torch.abs(y-.5))*(1+HARD*((y<=.2)*p.square()+(y>=.8)*(1-p).square()))
            w/=w.mean().clamp_min(1e-6)
        loss=(base*w).mean()
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
    scaler.step(opt); scaler.update(); sch.step(); run+=loss.item()

    if step%500==0:
        print(f"{step}/{steps} loss={run/500:.4f} speed={step*BS/(time.time()-started):.0f}/s")
        run=0
    if step%EVAL==0:
        score=macro(fast,predict(fast)); print("macro:",score)
        if score>best:
            best=score; torch.save({"model":core().state_dict(),"metric":score,"step":step},BEST)
            print("NEW BEST")
    if step>=steps:break

ck=torch.load(BEST,map_location="cpu",weights_only=True)
core().load_state_dict(ck["model"])
pred=predict_tta(va); score=macro(va,pred)
print(f"BEST STEP={ck['step']}, FAST={ck['metric']:.6f}, FULL TTA={score:.6f}")

OUT.mkdir(exist_ok=True)
core().save_pretrained(OUT); tok.save_pretrained(OUT)
json.dump({"llm_macro":score,"best_step":ck["step"],"max_len":MAXLEN},
          open(OUT/"metrics.json","w"),ensure_ascii=False,indent=2)
va.assign(predict=pred).to_parquet(WORK/"validation.parquet",index=False)
print(shutil.make_archive("/kaggle/working/ce_v2_stageA","zip",WORK,"model"))

['Tesla T4', 'Tesla T4']
split ready: 0.9 min
Polars pairs ready: 1.1 min
train=10,950,394, val=191,555


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

initial macro: 0.6438329362489459
500/42774 loss=0.4530 speed=549/s
1000/42774 loss=0.3743 speed=555/s
1500/42774 loss=0.3999 speed=555/s
2000/42774 loss=0.2572 speed=553/s
2500/42774 loss=0.3884 speed=557/s
3000/42774 loss=0.4236 speed=563/s
3500/42774 loss=0.4663 speed=564/s
4000/42774 loss=0.3901 speed=563/s
4500/42774 loss=0.3530 speed=558/s
5000/42774 loss=0.3751 speed=555/s
macro: 0.7188345092063256
NEW BEST
5500/42774 loss=0.3363 speed=541/s
6000/42774 loss=0.3178 speed=539/s
6500/42774 loss=0.4355 speed=541/s
7000/42774 loss=0.4246 speed=547/s
7500/42774 loss=0.3546 speed=548/s
8000/42774 loss=0.3329 speed=550/s
8500/42774 loss=0.4411 speed=549/s
9000/42774 loss=0.3951 speed=549/s
9500/42774 loss=0.3992 speed=549/s
10000/42774 loss=0.3675 speed=548/s
macro: 0.7233304634619304
NEW BEST
10500/42774 loss=0.4100 speed=539/s
11000/42774 loss=0.4433 speed=540/s
12000/42774 loss=0.4283 speed=541/s
12500/42774 loss=0.3652 speed=540/s
13000/42774 loss=0.3960 speed=541/s
13500/42774 loss